In [ ]:
学習済みモデルにに入力データを入力して変換した出力をファイルに保存するプログラム
正常に学習できているかを確認する

In [ ]:
import numpy as np
import tensorflow as tf


#グローバル変数
cir = 's344'  # 対象回路
tp_file = cir + '.vec'  # テストパターンファイル名
target_fault_num = 30  # 故障診断対象の故障の数⇒全ての故障の中からtarget_fault_num個をランダムに選択する
correct_output_file = 'correct_output/' + cir + '_correct_output'  # 正常な回路の出力ファイル
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + '分割正解データ' + '/' + cir + 'integrated_output'  # 統合後の正解データファイル
suplit_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_num'  # モデルの分割数が保存されたファイル
suplit_data_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_data_num'  # データの分割数が保存されたファイル
single_line_file = cir + '分割正解データ' + '/' + cir + 'single_line'  # 統合されていない信号線があるかが保存されたファイル
origin_output_data_file = 'model_output/' + cir + 'output/origin_output'  # モデルの出力を保存するファイル
change_output_data_file = 'model_output/' + cir + 'output/change_output'  # 閾値で変換したモデル出力を保存するファイル
input_data_num = None  # 1個のモデルにおける学習データ数
input_node_num = None  #入力層におけるノード数 初期値は8　学習結果によって変更
output_node_num = None #　出力層のノード数＝分割数による
num_models = None  # 学習させるモデルの数
model_folder = cir + 'sepmodel'  # 学習済みモデルを保存するフォルダ
fault_line_num = None  # 故障診断対象の回路における故障信号線の総数
fault_type_sum = 12  # 故障の種類の総数
# correct_value = [0.00, 0.25, 0.75, 1.00]  # 正解データの値・種類
# correct_value = [-1.00, -0.40, 0.40, 1.00]  # 出力層の活性化関数がtanhの場合の正解データの値（0~1ではなく、-1~1に変換する必要があるため） # 0->-1, 1->-0.5, 2->0.5, 3->1
correct_value = [0, 2, 4, 6]
# threshold = [0.02, 0.5, 0.98]  # ANNの出力値を変換するための閾値
# threshold = [-0.75, 0, 0.75]
threshold = [1, 3, 5]


processes = 8  # 並列処理のプロセス数

def mk_input_data():
    # 入力データファイルを開いてデータを読み込む
    with open(input_data_file) as f:
        lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]

    # print(lines)
    
    global input_data_num
    input_data_num = int(len(lines))      #学習データ数を設定。学習データ数は入力データの行数
    global input_node_num               #グローバル変数を書き換え
    input_node_num = int(len(lines[0]))      #入力ノード数を設定。入力ノード数は入力データの各行の要素数

    int_lines = [list(map(int, _)) for _ in lines]  #list要素の型をint型に変換

    return np.array(int_lines)


def mk_output_data(fname):
    # 正解データファイルを開いてデータを読み込む
    with open(fname) as f:
        lines = [_.replace("\n", "") for _ in f.readlines()]
    
    lines = [value.split(",") for value in lines] # カンマ区切りで各信号線をリストに格納
    
    # print("dsadsa")
    # print(lines[0])

    for i in range(len(lines)):
        for j in range(len(lines[i])):
            lines[i][j] = float(lines[i][j])
    
    # print("gaga")
    # print(lines[0])
    
    global output_node_num               #グローバル変数を書き換え
    output_node_num = int(len(lines[0]))      #出力ノード数を設定。出力ノード数は正解データの各行の要素数
    print("output_node_num:", output_node_num)



if __name__ == '__main__':
    
    # 学習済みの機械学習モデルに入力データを与えて、出力を取得する
    # 入力データをファイルから読み込む
    input_data = mk_input_data()

    # 分割モデルの数を取得
    with open(suplit_num_file, 'r') as f:
        num_models = int(f.readline())
  
    # データを何個づつ分割したのかを取得
    with open(suplit_data_num_file, 'r') as f:
        suplit_data_num = int(f.readline().replace("\n", ""))  # データの分割数を取得
        # print(suplit_data_num)


    # 統合されていない信号線があるかどうかを取得 0: 統合されてない信号線がない、1: 統合されていない信号線がある
    # 統合されていない信号線がある場合、最後のモデルの最後の出力ノードの値は統合されていない
    with open(single_line_file, 'r') as f:
        single_flag = int(f.readline().replace("\n", ""))

    g = open(origin_output_data_file, 'w')  # モデルの出力を保存するファイルを開く
    h = open(change_output_data_file, 'w')  # 閾値で変換したモデル出力を保存するファイルを開く
    for model_id in range(num_models):

        # モデルの読み込み
        with open(model_folder + '/' + cir + 'model_' + str(model_id) + '.tflite', 'rb') as f:  # モデルを読み込むファイルを開く
            tflite_model = f.read()

        # モデルの評価
        interpreter = tf.lite.Interpreter(model_content=tflite_model)  # TFLite形式のモデルを読み込む。保存されたTFLiteモデルをメモリに読み込み、推論を行う準備をするためのインタープリターを作成します。
        interpreter.allocate_tensors()  # #メモリを確保。モデルが使用するテンソル（データ構造）をメモリに割り当てます。TFLiteモデルをインタープリターにロードするだけでは、テンソルのメモリは割り当てられていません。この行を実行することで、モデルが推論に必要なメモリを確保します。

        input_details = interpreter.get_input_details()  # モデルの入力テンソルの詳細を取得。モデルの入力に関する詳細情報をリスト形式で返します。各要素は、入力テンソルの形状、データ型、名前などの情報を含む辞書です。
        output_details = interpreter.get_output_details()  # モデルの出力テンソルの詳細を取得。モデルの出力に関する詳細情報をリスト形式で返します。各要素は、出力テンソルの形状、データ型、名前などの情報を含む辞書です。

        correct_file = correct_data_file + str(model_id)  # 分割された正解データファイル名に変更　＝　s344分割正解データ/s344integrated_output + 番号
        mk_output_data(correct_file)  # 関数内でグローバル変数output_node_numを設定
  
        for i in range(input_data_num):
            
            input_shape = input_details[0]['shape']  # 入力データの形状を取得.先ほど取得したinput_detailsのリストの中から、形状情報を取得。input_details[0]['shape']は、入力テンソルの形状を取得するためのコードです。['shape']は、辞書内の shape キーを指定しています。

            reshape_input_data = np.reshape(input_data[i], input_shape)  # NumPy配列の形状を指定した形 (input_shape) に変更します。これにより、入力データの形状がモデルの入力テンソルの形状と一致します。


            # モデルの入力データを設定
            interpreter.set_tensor(input_details[0]['index'], np.array(reshape_input_data, dtype=np.float32))   # モデルの入力テンソルにデータを設定します。入力テンソルのインデックス、データを指定します。[0]['index']は、入力テンソルのインデックスを取得するためのコードです。

            # モデルの推論を実行
            interpreter.invoke()

            # モデルの出力データを取得
            output_data = interpreter.get_tensor(output_details[0]['index'])

            g.write(str(output_data[0]) + "\n")  # モデルの出力データをファイルに書き込む
            
            if (model_id == (num_models - 1)) and single_flag == 1:  # 最後のモデルで、統合されていない信号線がある場合
                for j in range(output_node_num):
                    if j < (output_node_num - 1):  # 最後の出力ノード以外の場合
                        if output_data[0][j] < threshold[0]:
                            output_data[0][j] = correct_value[0]
                        elif output_data[0][j] < threshold[1]:
                            output_data[0][j] = correct_value[1]
                        elif output_data[0][j] < threshold[2]:
                            output_data[0][j] = correct_value[2]
                        else:
                            output_data[0][j] = correct_value[3]
                    else:  # 統合されていない信号線がある場合、最後の出力ノードは統合されていない信号線の値を持つため、それは₀または1の値を持つため、0.5を閾値として0または1に変換する
                        if output_data[0][j] <= 0.5:
                            output_data[0][j] = correct_value[0]
                        else:
                            output_data[0][j] = correct_value[3]
                        # print(f"output_data[0][{j}]: {output_data[0][j]}")  # 最後の出力ノードの値を表示
            else:  # 最後のモデルで、統合されていない信号線がない場合、または最後のモデル以外の場合
                for j in range(output_node_num):
                    if output_data[0][j] < threshold[0]:
                        output_data[0][j] = correct_value[0]  # 0縮退故障の値
                    elif output_data[0][j] < threshold[1]:
                        output_data[0][j] = correct_value[1]  # 1縮退故障の値
                    elif output_data[0][j] < threshold[2]:
                        output_data[0][j] = correct_value[2]  # ブリッジ故障の値
                    else:
                        output_data[0][j] = correct_value[3]  # 正常な回路の値
            
            h.write(str(output_data[0]) + "\n")  # 閾値で変換したモデルの出力データをファイルに書き込む

    
    g.close()  # モデルの出力を保存するファイルを閉じる
    h.close()  # 閾値で変換したモデル出力を保存するファイルを閉じる
    print("モデルの出力を保存しました。")
            
           

output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
モデルの出力を保存しました。
